# Collatz-style rewriting for the 3-Decomposition Conjecture

This notebook develops a graph-rewriting programme for the **3-Decomposition Conjecture**:
every connected cubic graph should admit an edge partition into a spanning tree, a disjoint
union of cycles, and a matching.

It contains:

1. an exact tree-only defect formulation;
2. two proved non-increasing rewrite rules;
3. an exact matching-augmentation interpretation of defect transport;
4. exhaustive state-space tests on every connected simple cubic graph through ten vertices;
5. the current blossom-resolution proof target.

The finite computations are evidence, not a proof of the full conjecture.

In [1]:
# Colab-safe setup
import sys
import subprocess
import importlib.util

requirements = {
    "networkx": "networkx>=3.2",
    "pandas": "pandas>=2.0",
}
for module, package in requirements.items():
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

from collections import Counter, deque
from dataclasses import dataclass
import time

import networkx as nx
import pandas as pd
from networkx.algorithms.tree.mst import SpanningTreeIterator

print("NetworkX", nx.__version__)

NetworkX 3.6.1


## 1. Exact tree-only formulation

Let \(G\) be connected and cubic, and let \(T\) be a spanning tree. Put

\[
H_T=G-E(T).
\]

Every vertex has cotree degree \(3-\deg_T(v)\), so \(H_T\) has maximum degree two. Its
nontrivial components are paths and cycles. Define

\[
b(T)=\#\{\text{path components of }H_T\text{ with at least two edges}\}.
\]

Then

\[
\boxed{T\text{ extends to a 3-decomposition}\iff b(T)=0.}
\]

Indeed, cotree cycles form the cycle part, one-edge cotree paths form the matching, and
longer cotree paths are the only defects.

There is also an exact local identity. If \(e_{12}(T)\) denotes the number of cotree edges
whose endpoints have tree degrees one and two, then

\[
\boxed{e_{12}(T)=2b(T).}
\]

Each bad path has exactly two such end edges; no other cotree component has one.

In [2]:
class CubicRewriteSystem:
    """Bit-mask representation of spanning trees and cotree rewrites."""

    def __init__(self, G):
        G = nx.convert_node_labels_to_integers(G)
        if not nx.is_connected(G) or set(dict(G.degree()).values()) != {3}:
            raise ValueError("G must be a connected simple cubic graph.")
        self.G = G
        self.n = G.number_of_nodes()
        self.edges = [tuple(sorted(e)) for e in G.edges()]
        self.m = len(self.edges)
        self.index = {e: i for i, e in enumerate(self.edges)}
        self.full_mask = (1 << self.m) - 1

    def edge_index(self, u, v):
        return self.index[tuple(sorted((u, v)))]

    def tree_adjacency(self, mask):
        adj = [[] for _ in range(self.n)]
        for i, (u, v) in enumerate(self.edges):
            if (mask >> i) & 1:
                adj[u].append(v)
                adj[v].append(u)
        return adj

    def cotree_adjacency(self, mask):
        adj = [[] for _ in range(self.n)]
        for i, (u, v) in enumerate(self.edges):
            if not ((mask >> i) & 1):
                adj[u].append(v)
                adj[v].append(u)
        return adj

    def bad_paths(self, mask):
        H = self.cotree_adjacency(mask)
        seen = [False] * self.n
        result = []
        for start in range(self.n):
            if seen[start] or not H[start]:
                continue
            stack = [start]
            seen[start] = True
            component = []
            while stack:
                v = stack.pop()
                component.append(v)
                for w in H[v]:
                    if not seen[w]:
                        seen[w] = True
                        stack.append(w)
            edge_count = sum(len(H[v]) for v in component) // 2
            if edge_count < 2:
                continue
            ends = [v for v in component if len(H[v]) == 1]
            if len(ends) != 2:
                continue  # cycle component
            path = [ends[0]]
            previous = None
            current = ends[0]
            while current != ends[1]:
                nxt = next(w for w in H[current] if w != previous)
                path.append(nxt)
                previous, current = current, nxt
            result.append(path)
        return result

    def b(self, mask):
        return len(self.bad_paths(mask))

    def p2_only(self, mask):
        return all(len(path) == 3 for path in self.bad_paths(mask))

    def defect_edge_count(self, mask):
        T = self.tree_adjacency(mask)
        count = 0
        for i, (u, v) in enumerate(self.edges):
            if (mask >> i) & 1:
                continue
            if sorted((len(T[u]), len(T[v]))) == [1, 2]:
                count += 1
        return count

    def spanning_tree_masks(self):
        for tree in SpanningTreeIterator(self.G):
            mask = 0
            for u, v in tree.edges():
                mask |= 1 << self.edge_index(u, v)
            yield mask

    @staticmethod
    def tree_path(T, source, target):
        parent = {source: None}
        queue = deque([source])
        while queue:
            v = queue.popleft()
            if v == target:
                break
            for w in T[v]:
                if w not in parent:
                    parent[w] = v
                    queue.append(w)
        path = [target]
        v = target
        while v != source:
            v = parent[v]
            path.append(v)
        path.reverse()
        return path

    def basis_exchange_neighbors(self, mask):
        T = self.tree_adjacency(mask)
        neighbors = set()
        for i, (u, v) in enumerate(self.edges):
            if (mask >> i) & 1:
                continue
            path = self.tree_path(T, u, v)
            for a, b in zip(path, path[1:]):
                remove = self.edge_index(a, b)
                neighbors.add((mask | (1 << i)) & ~(1 << remove))
        return neighbors

    def endpoint_leaf_slides(self, mask):
        T = self.tree_adjacency(mask)
        before = self.b(mask)
        moves = set()
        for path in self.bad_paths(mask):
            for oriented in (path, list(reversed(path))):
                v0, v1 = oriented[0], oriented[1]
                assert len(T[v1]) == 1
                x = T[v1][0]
                add = self.edge_index(v0, v1)
                remove = self.edge_index(v1, x)
                nxt = (mask | (1 << add)) & ~(1 << remove)
                assert self.b(nxt) <= before
                moves.add(nxt)
        return moves

    def tail_peels(self, mask):
        T = self.tree_adjacency(mask)
        before = self.b(mask)
        moves = set()
        for path in self.bad_paths(mask):
            if len(path) < 4:
                continue
            for oriented in (path, list(reversed(path))):
                a, b = oriented[-3], oriented[-2]
                assert len(T[a]) == 1
                x = T[a][0]
                add = self.edge_index(a, b)
                remove = self.edge_index(a, x)
                nxt = (mask | (1 << add)) & ~(1 << remove)
                assert self.b(nxt) <= before
                moves.add(nxt)
        return moves

    def safe_moves(self, mask):
        return self.endpoint_leaf_slides(mask) | self.tail_peels(mask)

    def endpoint_transports(self, mask):
        T = self.tree_adjacency(mask)
        before = self.b(mask)
        moves = set()
        for path in self.bad_paths(mask):
            if len(path) != 3:
                continue
            for endpoint, center in ((path[0], path[1]), (path[2], path[1])):
                tree_path = self.tree_path(T, endpoint, center)
                x = tree_path[1]
                add = self.edge_index(endpoint, center)
                remove = self.edge_index(endpoint, x)
                nxt = (mask | (1 << add)) & ~(1 << remove)
                assert self.b(nxt) <= before
                moves.add(nxt)
        return moves

## 2. Two universal safe rewrites

### Endpoint leaf-slide

For a bad path \(v_0v_1\cdots v_k\), the internal vertex \(v_1\) is a tree leaf. If \(x\)
is its unique tree neighbour, use

\[
T'=T+v_0v_1-v_1x.
\]

This changes the parent of a tree leaf and satisfies \(b(T')\le b(T)\).

### Tail-peel

For a bad path \(v_0\cdots v_k\), \(k\ge3\), let
\(a=v_{k-2}\), \(b=v_{k-1}\), and let \(x\) be the unique tree neighbour of \(a\). Use

\[
T'=T+ab-ax.
\]

The final cotree edge becomes a harmless matching edge, while the remaining prefix is
redirected. Again \(b(T')\le b(T)\).

The rules may have neutral cycles, so a secondary termination argument is needed.

## 3. A strict clean-edge exit

Suppose the cotree has a defect \(a-b-c\). Select \(e=ab\). In \(H_T-e\), the edge
\(bc\) is harmless. If the fundamental tree path of \(e\) contains a tree edge \(f=uv\)
whose endpoints are both isolated in \(H_T-e\), then

\[
T'=T+e-f
\]

replaces the defect by the two disjoint matching edges \(bc\) and \(uv\). Therefore

\[
\boxed{b(T')=b(T)-1.}
\]

This is the clean fundamental-edge lemma.

## 4. Matching-augmentation interpretation

Orient a selected \(P_2\) defect as

\[
u_{-1}-c_0-u_0.
\]

Take all harmless one-edge cotree components and add the passive defect edge
\(u_{-1}c_0\). The result is a matching \(M_0\), and \(u_0\) is exposed.

A non-backtracking endpoint transport has the form

\[
u_{i-1}-c_i-u_i\longmapsto u_i-c_{i+1}-u_{i+1}.
\]

The corresponding matching update is

\[
M_{i+1}=M_i-\{c_{i+1}u_{i+1}\}+\{u_ic_{i+1}\}.
\]

Thus defect transport is exactly a matching flip along an alternating trail, with the extra
condition that every flip also preserves the spanning-tree complement.

If the next centre is a tree-degree-three vertex, it is exposed and the matching augments;
this is a strict decrease of \(b\). If the trajectory reaches another defect, the defects
collide and \(b\) decreases. If a vertex repeats first, an alternating blossom appears.

Hence the hard core is no longer an arbitrary dynamical system:

\[
\boxed{\text{augmentation, collision, or blossom}.}
\]

A blossom-resolution theorem compatible with the tree constraint is the principal open step.

## 5. Complete finite state spaces through ten vertices

There are \(1,2,5,19\) connected simple cubic graph isomorphism classes on
\(4,6,8,10\) vertices. The graph6 strings below give one representative of every class.

For every representative and every spanning tree, the notebook verifies:

1. the exact identity \(e_{12}=2b\);
2. endpoint slides, tail-peels, and \(P_2\) transports never increase \(b\);
3. no positive level component under all neutral basis exchanges is closed;
4. the safe subsystem reaches either \(b=0\) or a \(P_2\)-only state.

In [3]:
graph6_classes = {
    4: ["C~"],
    6: ["EZfG", "E]NG"],
    8: ["GqCilO", "GU`_zO", "GsSJHg", "GHqouC", "GLdBKW"],
    10: [
        "IAiKJ@PK_", "IG`iSaPK_", "IOKcmHOQG", "IGdV@a@BG",
        "IIEcJOIcG", "ICSlADo`O", "IKt@?gb`_", "IPPEGgXi?",
        "IlCCAKMJ?", "IqGSWoDWG", "ISHCq`DJ?", "ICWQZAccO",
        "I@oJAiad?", "IPPIDaQB_", "ILFGR?B_g", "IrAIH?XAo",
        "I?CwvBAw?", "IBYJ?IBcO", "IM@GpEg`O",
    ],
}

@dataclass
class ExhaustiveReport:
    graph: str
    vertices: int
    spanning_trees: int
    positive_neutral_components: int
    closed_positive_plateaux: int
    terminal_safe_components: int
    terminal_safe_components_without_target: int
    verified_safe_moves: int
    verified_endpoint_transports: int
    seconds: float


def analyze_graph(G, name):
    start = time.time()
    system = CubicRewriteSystem(G)
    states = list(system.spanning_tree_masks())
    state_set = set(states)
    bvals = {}

    neutral = nx.Graph()
    safe = nx.DiGraph()
    neutral.add_nodes_from(states)
    safe.add_nodes_from(states)

    safe_checks = 0
    transport_checks = 0

    for state in states:
        bvals[state] = system.b(state)
        assert system.defect_edge_count(state) == 2 * bvals[state]

    for state in states:
        for nxt in system.basis_exchange_neighbors(state):
            assert nxt in state_set
            if bvals[nxt] == bvals[state]:
                neutral.add_edge(state, nxt)

        for nxt in system.safe_moves(state):
            assert nxt in state_set
            assert bvals[nxt] <= bvals[state]
            safe.add_edge(state, nxt)
            safe_checks += 1

        if system.p2_only(state):
            for nxt in system.endpoint_transports(state):
                assert nxt in state_set
                assert bvals[nxt] <= bvals[state]
                transport_checks += 1

    positive_components = 0
    closed_plateaux = 0
    for component in nx.connected_components(neutral):
        level = bvals[next(iter(component))]
        assert all(bvals[state] == level for state in component)
        if level == 0:
            continue
        positive_components += 1
        has_strict_exit = any(
            bvals[nxt] < level
            for state in component
            for nxt in system.basis_exchange_neighbors(state)
        )
        if not has_strict_exit:
            closed_plateaux += 1

    terminal_safe = 0
    terminal_safe_without_target = 0
    for component in nx.strongly_connected_components(safe):
        terminal = not any(
            nxt not in component
            for state in component
            for nxt in safe.successors(state)
        )
        if not terminal:
            continue
        terminal_safe += 1
        target = any(
            bvals[state] == 0 or system.p2_only(state)
            for state in component
        )
        if not target:
            terminal_safe_without_target += 1

    return ExhaustiveReport(
        graph=name,
        vertices=G.number_of_nodes(),
        spanning_trees=len(states),
        positive_neutral_components=positive_components,
        closed_positive_plateaux=closed_plateaux,
        terminal_safe_components=terminal_safe,
        terminal_safe_components_without_target=terminal_safe_without_target,
        verified_safe_moves=safe_checks,
        verified_endpoint_transports=transport_checks,
        seconds=round(time.time() - start, 3),
    )

reports = []
for n, encodings in graph6_classes.items():
    for i, encoding in enumerate(encodings, start=1):
        G = nx.from_graph6_bytes(encoding.encode())
        reports.append(analyze_graph(G, f"n={n}, class={i}"))

report_table = pd.DataFrame([r.__dict__ for r in reports])
summary = (
    report_table.groupby("vertices", as_index=False)
    .agg(
        graph_classes=("graph", "count"),
        spanning_tree_states=("spanning_trees", "sum"),
        positive_neutral_components=("positive_neutral_components", "sum"),
        closed_positive_plateaux=("closed_positive_plateaux", "sum"),
        terminal_safe_components=("terminal_safe_components", "sum"),
        terminal_safe_without_target=("terminal_safe_components_without_target", "sum"),
        verified_safe_moves=("verified_safe_moves", "sum"),
        verified_endpoint_transports=("verified_endpoint_transports", "sum"),
        seconds=("seconds", "sum"),
    )
)
summary

,vertices,graph_classes,spanning_tree_states,positive_neutral_components,closed_positive_plateaux,terminal_safe_components,terminal_safe_without_target,verified_safe_moves,verified_endpoint_transports,seconds
0,4,1,16,1,0,4,0,48,0,0.003
1,6,2,156,9,0,18,0,552,48,0.032
2,8,5,1731,10,0,109,0,7360,644,0.505
3,10,19,28716,160,0,977,0,144160,12976,9.030


In [4]:
total_classes = len(reports)
total_states = sum(r.spanning_trees for r in reports)
total_closed = sum(r.closed_positive_plateaux for r in reports)
total_bad_safe = sum(r.terminal_safe_components_without_target for r in reports)
total_safe_moves = sum(r.verified_safe_moves for r in reports)
total_transports = sum(r.verified_endpoint_transports for r in reports)

print(f"Graph isomorphism classes: {total_classes}")
print(f"Spanning-tree states: {total_states:,}")
print(f"Verified safe rewrites: {total_safe_moves:,}")
print(f"Verified P2 endpoint transports: {total_transports:,}")
print(f"Closed positive neutral plateaux: {total_closed}")
print(f"Terminal safe SCCs without a target: {total_bad_safe}")

assert total_classes == 27
assert total_states == 30_619
assert total_closed == 0
assert total_bad_safe == 0

Graph isomorphism classes: 27
Spanning-tree states: 30,619
Verified safe rewrites: 152,120
Verified P2 endpoint transports: 13,668
Closed positive neutral plateaux: 0
Terminal safe SCCs without a target: 0


## 6. What the exhaustive result says

Across all 27 connected simple cubic graph classes through ten vertices and all 30,619
spanning-tree states:

\[
\boxed{\text{no positive neutral basis-exchange plateau is closed}.}
\]

This supports the **Monotone Exchange Conjecture**:

> From every spanning tree with \(b(T)>0\), a finite sequence of \(b\)-preserving basis
> exchanges reaches a strict decrease of \(b\).

The safe rules also satisfy

\[
\boxed{\text{every terminal safe-rule component contains }b=0
\text{ or a \(P_2\)-only state}.}
\]

Thus the complete small state space supports a three-layer architecture:

1. safe length normalization;
2. alternating-path transport of \(P_2\) defects;
3. blossom resolution and strict decrease.

A previously tempting universal claim that every plateau lies within three neutral steps of a
strict exit is **not** retained: larger random tests produced examples requiring up to five
neutral steps. The robust conjecture is existence of an exit, not a fixed radius-three bound.

## 7. Publication-readiness assessment

The work is not yet a publication-ready proof of a major open case. It has crossed the line
from analogy to a coherent research framework:

* an exact local defect statistic;
* two rigorous universal rewrites;
* strict clean-edge exits;
* a matching-augmentation interpretation;
* an explicit blossom obstruction;
* complete finite-state verification through ten vertices.

A genuinely substantial paper becomes available when at least one of the following is proved:

1. **Safe normalization theorem:** endpoint leaf-slides and tail-peels always reach a lower
   defect level or a \(P_2\)-only state.
2. **One-defect theorem:** every near-decomposition with a single \(P_2\) can be augmented to
   a true 3-decomposition.
3. **Blossom theorem:** every terminal alternating blossom admits a bounded rewrite that
   either contracts it or exposes a strict exit.
4. **Machine-checkable infinite-class theorem:** a finite DPO rewrite certificate proves
   termination on a new infinite graph class not covered by existing results.

The next mathematical target is the one-defect/blossom theorem, because it is sharp enough to
be publishable and directly attacks the only recurrent structure left by transport.

## 8. Reproducibility notes

The graph6 representatives are fixed in the notebook. Every reported state is regenerated from
scratch using NetworkX's spanning-tree iterator. All claimed rewrite inequalities and state-space
properties are enforced by assertions.

The finite verification does not use the known SAT verification of the original conjecture; it
tests stronger dynamical statements about **every** spanning tree of each graph.